# Build Coastal Chain (C)

這份 notebook 完成 **C 鏈（沿岸鏈）** 的第一版：

- 從 AOI 內 land polygons 建立 **保守碰撞層**（collision / ring）
- 用 **A2 smooth union** 產生「取點用」平滑輪廓
- 在平滑輪廓外推 `AVOID_KM` 得到 coast ring，並依 islands 規則過濾
- 沿 ring 等距取樣建立 **C_nodes**，並只做 **相鄰連線** 建立 **C_edges**
- 輸出 QA 統計與可視化

> 你只需要先設定 `SHP_PATH`（你的 land shapefile）以及 `ORIGIN_LL / DEST_LL`（或手動 bbox）。


In [11]:
# Cell 1 — Imports
import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from shapely.geometry import box, Polygon, MultiPolygon, LineString, Point
from shapely.ops import unary_union

import geopandas as gpd
from pyproj import CRS, Transformer

import folium


### Config

In [12]:
# Cell 2 — Config (請你先改這裡就能跑)
# --- Land shapefile path ---
# e.g. r"C:\data\ne_10m_land\ne_10m_land.shp"
SHP_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"  #

# --- AOI 設定：二選一 ---
# (A) 用 O/D 自動產 bbox
ORIGIN_LL = (129.033333,35.116667)  
DEST_LL   = (135.828056,33.468056)  

#| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
#| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
#| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
#| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
#| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
#| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
#| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
#| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
#| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
#| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
#| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
#| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
#| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
#| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |

PAD_DEG = 6.0              # bbox padding（degree）


# (B) 或者你直接手動給 bbox（優先於 O/D）
# BBOX_LL = (min_lon, min_lat, max_lon, max_lat)
BBOX_LL = None  # e.g. (109.4166, -24.6626, 144.7006, 39.9284)

# --- Land layers（碰撞用，保守）---
BUFFER_KM = 5.0
AVOID_KM = 15.0
COLLISION_SAFETY_KM = 0.25

# --- A2 smooth（取點基底，凶狠平滑）---
A2_SMOOTH_KM = 5.0
A2_TOL_KM = 8.0

# --- C 鏈密度（第一版起手值）---
C_STEP_KM = 20.0               # 沿岸鏈等距取樣步長
C_MAX_GAP_FACTOR = 2.0         # 任一相鄰段 > 2*C_STEP_KM 就補點（保護）
C_ROUND_DECIMALS = 5           # 去重 rounding（和你原 notebook 一致）

# --- Islands 規則（避免爆）---
ISLAND_AREA_MIN_KM2 = 20.0     # 小島面積門檻：低於這個就不納入 C 鏈（但 collision 仍會擋穿越）

# --- Debug / Draw ---
DEBUG_DRAW = True
DRAW_STEP_KM = 20.0


### Helpers

In [13]:
# Cell 3 — Geometry helpers (projection / bbox / sampling)

@dataclass
class AOIProjector:
    """Local metric projection centered at AOI centroid (Azimuthal Equidistant)."""
    crs_ll: CRS
    crs_m: CRS
    to_m: Transformer
    to_ll: Transformer

def make_aoi_bbox(origin_ll, dest_ll, pad_deg: float) -> tuple[float, float, float, float]:
    (lon1, lat1) = origin_ll
    (lon2, lat2) = dest_ll
    min_lon = min(lon1, lon2) - pad_deg
    max_lon = max(lon1, lon2) + pad_deg
    min_lat = min(lat1, lat2) - pad_deg
    max_lat = max(lat1, lat2) + pad_deg
    return (min_lon, min_lat, max_lon, max_lat)

def build_projector_from_bbox(bbox_ll: tuple[float, float, float, float]) -> AOIProjector:
    min_lon, min_lat, max_lon, max_lat = bbox_ll
    lon0 = (min_lon + max_lon) / 2.0
    lat0 = (min_lat + max_lat) / 2.0
    crs_ll = CRS.from_epsg(4326)
    # Local Azimuthal Equidistant, meters
    crs_m = CRS.from_proj4(f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs")
    to_m = Transformer.from_crs(crs_ll, crs_m, always_xy=True)
    to_ll = Transformer.from_crs(crs_m, crs_ll, always_xy=True)
    return AOIProjector(crs_ll=crs_ll, crs_m=crs_m, to_m=to_m, to_ll=to_ll)

def geom_to_m(geom, proj: AOIProjector):
    from shapely.ops import transform
    return transform(lambda x, y, z=None: proj.to_m.transform(x, y), geom)

def geom_to_ll(geom, proj: AOIProjector):
    from shapely.ops import transform
    return transform(lambda x, y, z=None: proj.to_ll.transform(x, y), geom)

def linestring_sample_points(line: LineString, step_m: float) -> list[Point]:
    """Sample points along a LineString at fixed step (meters)."""
    if line.length == 0:
        return []
    n = max(1, int(math.floor(line.length / step_m)))
    ds = line.length / n
    pts = [line.interpolate(i * ds) for i in range(n)]
    return pts


### Land loading and layers

In [14]:
# Cell 4 — Land loading & layers (ROBUST: make_valid / explode / precision snapping)

from pathlib import Path
from shapely.errors import GEOSException

try:
    # Shapely 2.x
    from shapely.validation import make_valid
except Exception:
    make_valid = None

try:
    # Shapely 2.x preferred
    from shapely import union_all, set_precision
except Exception:
    union_all = None
    set_precision = None


def load_polys_in_bbox(shp_path: str, bbox_ll: tuple[float, float, float, float]) -> list:
    """
    Load land geometries intersecting bbox using geopandas.
    Returns list of shapely geometries (WGS84).
    """
    shp_path = str(shp_path)
    if not Path(shp_path).exists():
        raise FileNotFoundError(f"SHP_PATH not found: {shp_path}")

    min_lon, min_lat, max_lon, max_lat = bbox_ll
    gdf = gpd.read_file(shp_path, bbox=(min_lon, min_lat, max_lon, max_lat))
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

    # explode multiparts (reduces union fragility)
    try:
        gdf = gdf.explode(index_parts=False, ignore_index=True)
    except TypeError:
        gdf = gdf.explode()

    # best-effort fix in WGS84
    def _fix_ll(g):
        if g is None or g.is_empty:
            return None
        gg = g
        try:
            if make_valid is not None:
                gg = make_valid(gg)
        except Exception:
            pass
        try:
            gg = gg.buffer(0)
        except Exception:
            pass
        return gg if (gg is not None and (not gg.is_empty)) else None

    gdf["geometry"] = gdf.geometry.apply(_fix_ll)
    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
    return list(gdf.geometry.values)


def build_land_layers(
    polys_ll: list,
    proj: AOIProjector,
    *,
    buffer_km: float,
    avoid_km: float,
    collision_safety_km: float,
    grid_size_m: float = 5.0,  # ✅ 重要：精度吸附（公尺），避免 TopologyException
) -> dict:
    """
    Build union/ring/collision layers in meters.

    Robustness strategy:
    - transform -> make_valid -> buffer(0)
    - set_precision(grid_size_m) to snap coordinates
    - union_all (preferred) with grid_size fallback
    """
    polys_m = []
    for g in polys_ll:
        gm = geom_to_m(g, proj)

        # fix invalid geometry in meters
        try:
            if make_valid is not None:
                gm = make_valid(gm)
        except Exception:
            pass
        try:
            gm = gm.buffer(0)
        except Exception:
            pass

        # precision snapping helps a lot on coastline datasets
        if set_precision is not None and grid_size_m and grid_size_m > 0:
            try:
                gm = set_precision(gm, grid_size_m)
            except Exception:
                pass

        if gm is not None and (not gm.is_empty):
            polys_m.append(gm)

    if len(polys_m) == 0:
        raise ValueError("No valid polygons after cleaning. Check SHP_PATH / AOI bbox.")

    # Union with fallback
    try:
        if union_all is not None:
            union_m = union_all(polys_m)
        else:
            union_m = unary_union(polys_m)
    except GEOSException:
        # Retry with union_all grid_size if available
        if union_all is None:
            raise
        union_m = union_all(polys_m, grid_size=grid_size_m)

    union_m = union_m.buffer(0)

    collision_prep_m = union_m.buffer(collision_safety_km * 1000.0)
    collision_m = collision_prep_m.buffer(0)

    ring_m = union_m.buffer(buffer_km * 1000.0).buffer(0)
    target_ring_m = union_m.buffer(avoid_km * 1000.0).buffer(0)

    return {
        "UNION_M": union_m,
        "COLLISION_PREP_M": collision_prep_m,
        "COLLISION_M": collision_m,
        "RING_M": ring_m,
        "TARGET_RING_M": target_ring_m,
    }


### A2 smooth union features

In [15]:
# Cell 5 — A2 smooth union for features (use UNION_M from layers, no re-union)

def smooth_union_for_features_from_union(union_m, *, smooth_km: float, tol_km: float):
    """
    A2 smooth on already-unioned geometry:
    union_m -> buffer(+r) -> buffer(-r) -> simplify(tol) -> buffer(0)
    """
    r = smooth_km * 1000.0
    tol = tol_km * 1000.0

    sm = union_m.buffer(r).buffer(-r)
    sm = sm.simplify(tol, preserve_topology=True)
    sm = sm.buffer(0)
    return sm



### Build coast rings from A2 smooth union 

In [16]:
# Cell 6 — Build coast rings from A2 smooth union (buffer AVOID_KM, filter islands)

from shapely.geometry import GeometryCollection

def _iter_polygons(geom):
    if geom.is_empty:
        return []
    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return list(geom.geoms)
    if isinstance(geom, GeometryCollection):
        polys = []
        for g in geom.geoms:
            polys.extend(_iter_polygons(g))
        return polys
    return []

def build_coast_rings_smooth(union_smooth_m, *, avoid_km: float, island_area_min_km2: float):
    ring_base_m = union_smooth_m.buffer(avoid_km * 1000.0).buffer(0)
    polys = _iter_polygons(ring_base_m)
    if not polys:
        return ring_base_m, [], pd.DataFrame(columns=["ring_id","area_km2","length_km","is_mainland"])

    areas = np.array([p.area for p in polys], dtype=float)
    main_idx = int(np.argmax(areas))

    rings = []
    rows = []
    ring_id = 0
    for i, p in enumerate(polys):
        area_km2 = p.area / 1e6
        length_km = p.exterior.length / 1000.0
        is_mainland = (i == main_idx)
        keep = is_mainland or (area_km2 >= island_area_min_km2)
        if not keep:
            continue
        rings.append(p.exterior)
        rows.append({
            "ring_id": ring_id,
            "area_km2": area_km2,
            "length_km": length_km,
            "is_mainland": is_mainland
        })
        ring_id += 1

    rings_df = pd.DataFrame(rows).sort_values(["is_mainland","area_km2"], ascending=[False, False]).reset_index(drop=True)
    return ring_base_m, rings, rings_df


### Build C-nodes + C_edges

In [17]:
# Cell 7 — Build C_nodes + C_edges

def build_C_chain_from_rings(
    rings_m: list[LineString],
    proj: AOIProjector,
    *,
    c_step_km: float,
    round_decimals: int,
):
    c_step_m = c_step_km * 1000.0
    C_nodes_rows = []
    C_edges_rows = []

    c_id = 0
    for ring_id, ring in enumerate(rings_m):
        if ring.is_empty or ring.length <= 0:
            continue

        pts_m = linestring_sample_points(ring, step_m=c_step_m)

        # Convert + de-dup
        seen = set()
        pts_kept = []
        for p in pts_m:
            x, y = p.x, p.y
            lon, lat = proj.to_ll.transform(x, y)
            key = (round(lon, round_decimals), round(lat, round_decimals))
            if key in seen:
                continue
            seen.add(key)
            pts_kept.append((lon, lat, x, y))

        if len(pts_kept) < 3:
            # fallback: add a few vertices
            coords = list(ring.coords)
            for (x, y) in [coords[0], coords[len(coords)//2], coords[-1]]:
                lon, lat = proj.to_ll.transform(x, y)
                key = (round(lon, round_decimals), round(lat, round_decimals))
                if key not in seen:
                    seen.add(key)
                    pts_kept.append((lon, lat, x, y))

        # arclength s_km along sampled order
        s_km = 0.0
        ring_c_ids = []
        for i, (lon, lat, x, y) in enumerate(pts_kept):
            if i > 0:
                x0, y0 = pts_kept[i-1][2], pts_kept[i-1][3]
                s_km += math.hypot(x - x0, y - y0) / 1000.0
            C_nodes_rows.append({
                "c_id": c_id,
                "ring_id": ring_id,
                "lon": lon,
                "lat": lat,
                "x_m": x,
                "y_m": y,
                "s_km": s_km,
            })
            ring_c_ids.append(c_id)
            c_id += 1

        # Adjacent edges
        for i in range(len(ring_c_ids) - 1):
            u = ring_c_ids[i]
            v = ring_c_ids[i+1]
            xu, yu = C_nodes_rows[u]["x_m"], C_nodes_rows[u]["y_m"]
            xv, yv = C_nodes_rows[v]["x_m"], C_nodes_rows[v]["y_m"]
            length_km = math.hypot(xv - xu, yv - yu) / 1000.0
            C_edges_rows.append({"u": u, "v": v, "length_km": length_km, "edge_type": "C-C"})

        # Close ring
        if len(ring_c_ids) >= 3:
            u = ring_c_ids[-1]
            v = ring_c_ids[0]
            xu, yu = C_nodes_rows[u]["x_m"], C_nodes_rows[u]["y_m"]
            xv, yv = C_nodes_rows[v]["x_m"], C_nodes_rows[v]["y_m"]
            length_km = math.hypot(xv - xu, yv - yu) / 1000.0
            C_edges_rows.append({"u": u, "v": v, "length_km": length_km, "edge_type": "C-C"})

    C_nodes = pd.DataFrame(C_nodes_rows)
    C_edges = pd.DataFrame(C_edges_rows)
    return C_nodes, C_edges

def compute_ring_gap_stats(C_nodes: pd.DataFrame) -> pd.DataFrame:
    out = []
    for ring_id, grp in C_nodes.groupby("ring_id"):
        grp = grp.sort_values("s_km")
        xy = grp[["x_m","y_m"]].values
        if len(xy) < 3:
            continue
        gaps = np.sqrt(np.sum((xy[1:] - xy[:-1])**2, axis=1)) / 1000.0
        out.append({
            "ring_id": ring_id,
            "n_nodes": len(grp),
            "gap_km_mean": float(gaps.mean()) if len(gaps) else np.nan,
            "gap_km_max": float(gaps.max()) if len(gaps) else np.nan,
        })
    return pd.DataFrame(out).sort_values("gap_km_max", ascending=False).reset_index(drop=True)


### Run pipeline

In [18]:
# Cell 8 — Run pipeline

if BBOX_LL is None:
    BBOX_LL = make_aoi_bbox(ORIGIN_LL, DEST_LL, PAD_DEG)

print("[AOI] bbox_ll =", BBOX_LL)
proj = build_projector_from_bbox(BBOX_LL)

polys_ll = load_polys_in_bbox(SHP_PATH, BBOX_LL)
print(f"[LAND] loaded polys = {len(polys_ll)}");

layers = build_land_layers(
    polys_ll, proj,
    buffer_km=BUFFER_KM,
    avoid_km=AVOID_KM,
    collision_safety_km=COLLISION_SAFETY_KM,
    grid_size_m=25.0,
)
print("[LAYERS] built: UNION_M, COLLISION_M, RING_M, TARGET_RING_M");

union_smooth_m = smooth_union_for_features_from_union(
    layers["UNION_M"],
    smooth_km=A2_SMOOTH_KM,
    tol_km=A2_TOL_KM,
)

print("[A2] smooth union done");

ring_base_m, coast_rings_smooth_m, rings_df = build_coast_rings_smooth(
    union_smooth_m,
    avoid_km=AVOID_KM,
    island_area_min_km2=ISLAND_AREA_MIN_KM2,
)
print(f"[RINGS] kept rings = {len(coast_rings_smooth_m)}");
display(rings_df.head(20))

C_nodes, C_edges = build_C_chain_from_rings(
    coast_rings_smooth_m,
    proj,
    c_step_km=C_STEP_KM,
    round_decimals=C_ROUND_DECIMALS,
)
print(f"[C] nodes={len(C_nodes)} edges={len(C_edges)}");
display(C_nodes.head())
display(C_edges.head())

gap_stats = compute_ring_gap_stats(C_nodes)
display(gap_stats.head(20))


[AOI] bbox_ll = (123.033333, 27.468055999999997, 141.828056, 41.116667)
[LAND] loaded polys = 6689
[LAYERS] built: UNION_M, COLLISION_M, RING_M, TARGET_RING_M
[A2] smooth union done
[RINGS] kept rings = 1140


,ring_id,area_km2,length_km,is_mainland
0,0,1.711050e+08,226034.691632,True
1,526,1.188189e+08,124419.417969,False
2,319,3.187279e+07,66378.861917,False
3,524,9.662557e+06,22193.093986,False
4,618,3.657840e+06,15707.379131,False
5,1113,1.109608e+06,5865.395794,False
6,550,1.014921e+06,8872.092924,False
7,837,8.854210e+05,5353.874024,False
8,914,5.941341e+05,7168.212386,False
9,644,5.455102e+05,8424.039270,False


[C] nodes=37483 edges=37483


,c_id,ring_id,lon,lat,x_m,y_m,s_km
0,0,0,-40.914423,-21.863850,-8.176175e+06,1.657825e+07,0.000000
1,1,0,-40.955807,-21.978353,-8.193645e+06,1.658450e+07,18.554715
2,2,0,-41.022949,-22.105333,-8.191674e+06,1.660293e+07,37.091987
3,3,0,-41.047383,-22.121326,-8.175072e+06,1.661404e+07,57.064529
4,4,0,-41.070331,-22.134050,-8.158215e+06,1.662480e+07,77.065829


,u,v,length_km,edge_type
0,0,1,18.554715,C-C
1,1,2,18.537272,C-C
2,2,3,19.972541,C-C
3,3,4,20.001300,C-C
4,4,5,20.001300,C-C


,ring_id,n_nodes,gap_km_mean,gap_km_max
0,310,5,22.283694,23.704921
1,40,5,22.289753,23.656577
2,702,5,22.329752,23.347205
3,335,5,22.316287,23.242160
4,65,5,22.035902,23.199594
5,431,5,22.039322,23.190783
6,1013,6,21.623395,23.087670
7,180,5,22.048754,23.067618
8,276,6,21.609244,23.010256
9,119,5,21.763405,22.976937


### Visualization

In [23]:
# Cell 9 — Visualization (Folium + LayerControl + open in browser)

import webbrowser
from pathlib import Path
import folium

min_lon, min_lat, max_lon, max_lat = BBOX_LL
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

m = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron")

# ----------------------------
# Layer groups
# ----------------------------
fg_bbox = folium.FeatureGroup(name="AOI bbox", show=True)
fg_rings = folium.FeatureGroup(name="A2 coast rings (sampled)", show=True)
fg_c_nodes = folium.FeatureGroup(name="C nodes (sampled)", show=True)

# (Optional) land layers outline (may be heavy)
fg_union = folium.FeatureGroup(name="UNION_M outline (downsampled)", show=False)
fg_collision = folium.FeatureGroup(name="COLLISION_M outline (downsampled)", show=False)

# ----------------------------
# Helpers
# ----------------------------
def add_linestring_ll(line_m: LineString, proj: AOIProjector, group, *, step_km=20.0, weight=2, opacity=0.7):
    if line_m is None or line_m.is_empty:
        return
    step_m = max(1000.0, step_km * 1000.0)
    pts = linestring_sample_points(line_m, step_m=step_m)
    latlons = []
    for p in pts:
        lon, lat = proj.to_ll.transform(p.x, p.y)
        latlons.append((lat, lon))
    if len(latlons) >= 2:
        folium.PolyLine(latlons, weight=weight, opacity=opacity).add_to(group)

def add_polygon_outline_ll(poly_m, proj: AOIProjector, group, *, step_km=30.0, weight=2, opacity=0.5):
    # Draw only exteriors, downsampled, to avoid super heavy HTML
    if poly_m is None or poly_m.is_empty:
        return
    for p in _iter_polygons(poly_m):
        add_linestring_ll(p.exterior, proj, group, step_km=step_km, weight=weight, opacity=opacity)

# ----------------------------
# AOI bbox layer
# ----------------------------
folium.Rectangle(
    bounds=[(min_lat, min_lon), (max_lat, max_lon)],
    fill=False,
).add_to(fg_bbox)

fg_bbox.add_to(m)

# ----------------------------
# Rings layer (limit count to keep HTML light)
# ----------------------------
# --- pick rings to draw: prefer rings whose centroid is inside AOI bbox; fallback to top area ---
min_lon, min_lat, max_lon, max_lat = BBOX_LL

ring_by_id = {i: r for i, r in enumerate(coast_rings_smooth_m)}

def ring_centroid_ll(ring_m):
    c = ring_m.centroid
    lon, lat = proj.to_ll.transform(c.x, c.y)
    return lon, lat

# add centroid to rings_df (only once)
if "centroid_lon" not in rings_df.columns:
    lons, lats = [], []
    for rid in rings_df["ring_id"]:
        lon, lat = ring_centroid_ll(ring_by_id[rid])
        lons.append(lon); lats.append(lat)
    rings_df["centroid_lon"] = lons
    rings_df["centroid_lat"] = lats

in_aoi = rings_df[
    (rings_df["centroid_lon"] >= min_lon) & (rings_df["centroid_lon"] <= max_lon) &
    (rings_df["centroid_lat"] >= min_lat) & (rings_df["centroid_lat"] <= max_lat)
].copy()

# choose K rings
K = 60
pick_df = (in_aoi if len(in_aoi) else rings_df).sort_values("area_km2", ascending=False).head(K)
rings_to_draw = [ring_by_id[rid] for rid in pick_df["ring_id"].tolist()]

# --- dateline-safe polyline drawing helper ---
def _norm_lon(lon):
    return ((lon + 180) % 360) - 180

def _split_dateline(latlons):
    # latlons: [(lat, lon), ...] with lon normalized
    segs = []
    seg = []
    for lat, lon in latlons:
        if not seg:
            seg = [(lat, lon)]
            continue
        prev_lon = seg[-1][1]
        if abs(lon - prev_lon) > 180:
            if len(seg) >= 2:
                segs.append(seg)
            seg = [(lat, lon)]
        else:
            seg.append((lat, lon))
    if len(seg) >= 2:
        segs.append(seg)
    return segs

def add_linestring_ll_safe(line_m, group, step_km=20.0, weight=2, opacity=0.8):
    if line_m.is_empty:
        return
    pts = linestring_sample_points(line_m, step_m=max(1000.0, step_km * 1000.0))
    latlons = []
    for p in pts:
        lon, lat = proj.to_ll.transform(p.x, p.y)
        lon = _norm_lon(lon)
        latlons.append((lat, lon))
    for seg in _split_dateline(latlons):
        folium.PolyLine(seg, weight=weight, opacity=opacity).add_to(group)

# --- draw selected rings ---
for ring in rings_to_draw:
    add_linestring_ll_safe(ring, fg_rings, step_km=DRAW_STEP_KM, weight=2, opacity=0.8)

fg_rings.add_to(m)

# ----------------------------
# C nodes layer (sample to keep light)
# ----------------------------
if len(C_nodes):
    sample_n = min(len(C_nodes), 4000)
    sample = C_nodes.sample(sample_n, random_state=0) if len(C_nodes) > sample_n else C_nodes
    for _, r in sample.iterrows():
        folium.CircleMarker(
            location=(r["lat"], r["lon"]),
            radius=2,
            weight=0,
            fill=True,
            fill_opacity=0.7,
        ).add_to(fg_c_nodes)

fg_c_nodes.add_to(m)

# ----------------------------
# Optional: outline layers (can be heavy; default off)
# ----------------------------
# Note: these are metric geometries; we just draw outlines, downsampled
try:
    add_polygon_outline_ll(layers["UNION_M"], proj, fg_union, step_km=40.0, weight=2, opacity=0.35)
    fg_union.add_to(m)
except Exception as e:
    print("[warn] UNION_M outline skipped:", e)

try:
    add_polygon_outline_ll(layers["COLLISION_M"], proj, fg_collision, step_km=40.0, weight=2, opacity=0.35)
    fg_collision.add_to(m)
except Exception as e:
    print("[warn] COLLISION_M outline skipped:", e)

# ----------------------------
# Layer control + save + open
# ----------------------------
folium.LayerControl(collapsed=False).add_to(m)

OUT_DIR = Path("./outputs_c_chain")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_html = OUT_DIR / "c_chain_layers.html"
m.save(str(out_html))

print("Saved map to:", out_html.resolve())
webbrowser.open(out_html.resolve().as_uri())


Saved map to: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs_c_chain\c_chain_layers.html


True

### Save outputs

In [20]:
# Cell 10 — Save outputs (optional)
OUT_DIR = Path("./outputs_c_chain")
OUT_DIR.mkdir(parents=True, exist_ok=True)

C_nodes.to_parquet(OUT_DIR / "C_nodes.parquet", index=False)
C_edges.to_parquet(OUT_DIR / "C_edges.parquet", index=False)
rings_df.to_parquet(OUT_DIR / "rings_df.parquet", index=False)

print("Saved:", OUT_DIR.resolve())


Saved: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs_c_chain


# Feature nodes

## Config

In [24]:
# === Feature nodes (F) ===
F_SAMPLE_STEP_KM = 10.0     # 沿 A2 union exterior 取樣步長（比 C 稍密）
F_MIN_TURN_DEG = 25.0       # 轉角門檻：越大越少點
F_NMS_RADIUS_KM = 30.0      # NMS 抑制半徑：避免同一岬角留一串點
F_MAX_PER_POLY = 80         # 每條 exterior(主陸/大島)最多保留幾個 feature
F_MAX_TOTAL = 400           # AOI 全部 feature 上限

# === Gate-A (coverage first, no sea yet) ===
GATE_SPACING_KM = 200.0     # 沿每條 C-ring 每隔多少 km 放一個 gate（coverage）
GATE_SNAP_TO_F_KM = 40.0    # 若 gate 附近有 F，距離 <= 40km 就吸附到該 F
GATE_ROUND_DECIMALS = 5     # gate 去重 rounding

# Debug
DEBUG_F_GATE = True


## F nodes

In [25]:
import numpy as np
import pandas as pd
import math
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.geometry import GeometryCollection

def _iter_polygons_any(geom):
    """Yield polygons from Polygon/MultiPolygon/GeometryCollection/etc."""
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return list(geom.geoms)
    if isinstance(geom, GeometryCollection):
        polys = []
        for g in geom.geoms:
            polys.extend(_iter_polygons_any(g))
        return polys
    # fallback: try to access .geoms
    if hasattr(geom, "geoms"):
        polys = []
        for g in geom.geoms:
            polys.extend(_iter_polygons_any(g))
        return polys
    return []

def _sample_line_xy(line: LineString, step_km: float):
    """Return sampled points as Nx2 array (x,y) in meters."""
    step_m = max(1000.0, step_km * 1000.0)
    pts = linestring_sample_points(line, step_m=step_m)
    if not pts:
        return np.zeros((0,2), dtype=float)
    return np.array([[p.x, p.y] for p in pts], dtype=float)

def _turn_angles_deg(xy: np.ndarray) -> np.ndarray:
    """
    Compute turning angle at each interior point using 3 consecutive points.
    Straight line -> ~0 deg, sharp corner -> larger.
    """
    n = len(xy)
    if n < 3:
        return np.zeros(n, dtype=float)
    ang = np.zeros(n, dtype=float)
    for i in range(1, n-1):
        a = xy[i] - xy[i-1]
        b = xy[i+1] - xy[i]
        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)
        if na < 1e-9 or nb < 1e-9:
            ang[i] = 0.0
            continue
        cosv = float(np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0))
        ang[i] = math.degrees(math.acos(cosv))
    # endpoints keep 0
    return ang

def _nms_points_xy(xy: np.ndarray, score: np.ndarray, radius_km: float, max_keep: int):
    """
    Greedy NMS: keep high-score points, suppress any within radius_km.
    """
    if len(xy) == 0:
        return np.array([], dtype=int)
    r2 = (radius_km * 1000.0) ** 2
    order = np.argsort(-score)  # desc
    keep = []
    kept_xy = []
    for idx in order:
        if score[idx] <= 0:
            break
        p = xy[idx]
        ok = True
        for q in kept_xy:
            if np.sum((p - q) ** 2) <= r2:
                ok = False
                break
        if ok:
            keep.append(idx)
            kept_xy.append(p)
            if len(keep) >= max_keep:
                break
    return np.array(keep, dtype=int)

def extract_F_nodes_from_union_smooth(
    union_smooth_m,
    proj: AOIProjector,
    *,
    sample_step_km: float,
    min_turn_deg: float,
    nms_radius_km: float,
    max_per_poly: int,
    max_total: int,
    island_area_min_km2: float = None,
):
    """
    Extract feature nodes from A2 smooth union exterior(s).
    - sample exterior line
    - compute turn angle as score
    - threshold + NMS
    """
    polys = _iter_polygons_any(union_smooth_m)
    if island_area_min_km2 is not None and len(polys) > 1:
        # keep main + big islands
        areas = np.array([p.area for p in polys], dtype=float)
        main_idx = int(np.argmax(areas))
        kept = []
        for i, p in enumerate(polys):
            area_km2 = p.area / 1e6
            if i == main_idx or area_km2 >= island_area_min_km2:
                kept.append(p)
        polys = kept

    rows = []
    f_id = 0

    for poly_id, p in enumerate(polys):
        line = p.exterior
        xy = _sample_line_xy(line, sample_step_km)
        if len(xy) < 3:
            continue
        score = _turn_angles_deg(xy)

        # threshold
        score_thr = np.where(score >= min_turn_deg, score, 0.0)

        # NMS
        keep_idx = _nms_points_xy(xy, score_thr, radius_km=nms_radius_km, max_keep=max_per_poly)

        for i in keep_idx:
            x, y = float(xy[i,0]), float(xy[i,1])
            lon, lat = proj.to_ll.transform(x, y)
            rows.append({
                "f_id": f_id,
                "poly_id": poly_id,
                "lon": lon,
                "lat": lat,
                "x_m": x,
                "y_m": y,
                "score": float(score[i]),
            })
            f_id += 1

    F_nodes = pd.DataFrame(rows)

    # global cap
    if len(F_nodes) > max_total:
        F_nodes = F_nodes.sort_values("score", ascending=False).head(max_total).reset_index(drop=True)
        F_nodes["f_id"] = np.arange(len(F_nodes), dtype=int)

    return F_nodes


## nodes + Gate-A（coverage + snap-to-F）

In [30]:
# Cell C — Run F nodes + Gate-A (slim v1: length-based coverage)

import numpy as np
import pandas as pd
import math

# Gate-A slimming v1
GATE_MIN_RING_LENGTH_KM = 300.0
GATE_SHORT_RING_NO_GATE_KM = 150.0
GATE_SHORT_RING_ONE_GATE_KM = 400.0

# === 1) Extract F nodes from union_smooth_m exterior(s)
F_nodes = extract_F_nodes_from_union_smooth(
    union_smooth_m,
    proj,
    sample_step_km=F_SAMPLE_STEP_KM,
    min_turn_deg=F_MIN_TURN_DEG,
    nms_radius_km=F_NMS_RADIUS_KM,
    max_per_poly=F_MAX_PER_POLY,
    max_total=F_MAX_TOTAL,
    island_area_min_km2=ISLAND_AREA_MIN_KM2,   # ✅延用 islands 過濾門檻
)

print(f"[F] nodes={len(F_nodes)}  (step={F_SAMPLE_STEP_KM}km, min_turn={F_MIN_TURN_DEG}deg, nms={F_NMS_RADIUS_KM}km)")
display(F_nodes.head(10))


# === 2) Gate-A (slim v1): coverage only for rings with enough coastline length
# Rules (v1):
# - if ring_length_km < GATE_MIN_RING_LENGTH_KM: skip coverage (0 gates)
# - else:
#     - if L < GATE_SHORT_RING_NO_GATE_KM: 0 gates (rare since min_length already filters)
#     - elif L < GATE_SHORT_RING_ONE_GATE_KM: 1 gate
#     - else: n_gate = floor(L / GATE_SPACING_KM) (and ensure >=2)

def build_gate_A_from_C_and_F_v1(
    C_nodes: pd.DataFrame,
    F_nodes: pd.DataFrame,
    rings_df: pd.DataFrame,
    *,
    gate_spacing_km: float,
    snap_to_f_km: float,
    round_decimals: int,
    min_ring_length_km: float,
    short_no_gate_km: float,
    short_one_gate_km: float,
    enforce_min_gates_for_long_rings: bool = True,
) -> pd.DataFrame:
    # map ring_id -> length_km
    ring_len = dict(zip(rings_df["ring_id"].astype(int), rings_df["length_km"].astype(float)))

    # Prepack F in numpy arrays (gate count is small -> brute force ok)
    Fx = F_nodes["x_m"].to_numpy(dtype=float) if len(F_nodes) else np.zeros((0,), dtype=float)
    Fy = F_nodes["y_m"].to_numpy(dtype=float) if len(F_nodes) else np.zeros((0,), dtype=float)
    Fr2 = (snap_to_f_km * 1000.0) ** 2

    def nearest_F(x, y):
        if len(Fx) == 0:
            return None
        dx = Fx - x
        dy = Fy - y
        d2 = dx*dx + dy*dy
        j = int(np.argmin(d2))
        if float(d2[j]) <= Fr2:
            return j
        return None

    rows = []
    seen = set()
    g_id = 0

    # Stats
    kept_rings = 0
    skipped_short = 0
    ring_gate_counts = {}

    for ring_id, grp in C_nodes.groupby("ring_id"):
        ring_id = int(ring_id)

        L = ring_len.get(ring_id, None)
        if L is None:
            continue

        # Coverage eligibility (main slimming knob)
        if L < float(min_ring_length_km):
            skipped_short += 1
            continue

        # Decide n_gate by coastline length L (km)
        if L < float(short_no_gate_km):
            # typically won't happen because min_ring_length_km >= 300 in v1
            continue
        elif L < float(short_one_gate_km):
            n_gate = 1
        else:
            n_gate = int(math.floor(L / float(gate_spacing_km)))
            if enforce_min_gates_for_long_rings:
                n_gate = max(n_gate, 2)
            else:
                n_gate = max(n_gate, 1)

        grp = grp.sort_values("s_km").reset_index(drop=True)
        if len(grp) < 2:
            continue

        s_max = float(grp["s_km"].iloc[-1])
        if s_max <= 0:
            continue

        # Coverage targets along arclength
        s_targets = np.linspace(0.0, s_max, n_gate, endpoint=False)
        s_arr = grp["s_km"].to_numpy(dtype=float)

        n_added = 0
        for s_t in s_targets:
            idx = int(np.argmin(np.abs(s_arr - s_t)))
            c = grp.iloc[idx]
            x, y = float(c["x_m"]), float(c["y_m"])
            lon, lat = float(c["lon"]), float(c["lat"])

            src = "C"
            f_idx = nearest_F(x, y)
            if f_idx is not None:
                f = F_nodes.iloc[f_idx]
                x, y = float(f["x_m"]), float(f["y_m"])
                lon, lat = float(f["lon"]), float(f["lat"])
                src = "F"

            key = (round(lon, round_decimals), round(lat, round_decimals), ring_id)
            if key in seen:
                continue
            seen.add(key)

            rows.append({
                "g_id": g_id,
                "ring_id": ring_id,
                "lon": lon,
                "lat": lat,
                "x_m": x,
                "y_m": y,
                "source": src,
                "s_target_km": float(s_t),
                "ring_length_km": float(L),
                "n_gate_plan": int(n_gate),
            })
            g_id += 1
            n_added += 1

        if n_added > 0:
            kept_rings += 1
            ring_gate_counts[ring_id] = n_added

    Gate_A = pd.DataFrame(rows)

    # QA prints
    print(f"[Gate-A v1] rings eligible length>={min_ring_length_km}km: {kept_rings} (skipped_short={skipped_short})")
    if len(Gate_A):
        print(f"[Gate-A v1] candidates={len(Gate_A)}  (spacing={gate_spacing_km}km, snap_to_F={snap_to_f_km}km)")
        print("[Gate-A v1] source counts:")
        print(Gate_A["source"].value_counts(dropna=False))
        # per-ring distribution
        s = pd.Series(ring_gate_counts)
        print("[Gate-A v1] per-ring gate count (eligible rings) describe:")
        print(s.describe())
    else:
        print("[Gate-A v1] candidates=0 (check thresholds)")

    return Gate_A


Gate_A = build_gate_A_from_C_and_F_v1(
    C_nodes,
    F_nodes,
    rings_df,
    gate_spacing_km=GATE_SPACING_KM,
    snap_to_f_km=GATE_SNAP_TO_F_KM,
    round_decimals=GATE_ROUND_DECIMALS,
    min_ring_length_km=GATE_MIN_RING_LENGTH_KM,
    short_no_gate_km=GATE_SHORT_RING_NO_GATE_KM,
    short_one_gate_km=GATE_SHORT_RING_ONE_GATE_KM,
    enforce_min_gates_for_long_rings=True,
)

display(Gate_A.head(10))


[F] nodes=400  (step=10.0km, min_turn=25.0deg, nms=30.0km)


,f_id,poly_id,lon,lat,x_m,y_m,score
0,0,250,173.937726,0.352922,4.853531e+06,-3.032797e+06,179.449115
1,1,8,-51.820877,-31.797262,1.570143e+07,1.162306e+07,179.209734
2,2,0,-48.477208,-27.154036,2.050475e+06,1.909831e+07,178.277916
3,3,351,-164.945408,54.113437,3.735786e+06,3.705677e+06,178.056618
4,4,0,-74.503541,-51.718073,1.091469e+07,-1.318453e+07,177.744855
5,5,0,-43.728442,-23.004223,-5.475225e+06,1.788058e+07,177.733942
6,6,1025,-38.084665,-54.003955,-4.779025e+06,-1.703484e+07,177.569382
7,7,9,-51.665367,-31.765590,1.540371e+07,1.202973e+07,177.292775
8,8,0,-73.419406,-42.563884,1.522016e+07,-8.830909e+06,176.979763
9,9,0,-51.337282,-30.670889,1.248047e+07,1.495492e+07,176.858403


[Gate-A v1] rings eligible length>=300.0km: 187 (skipped_short=953)
[Gate-A v1] candidates=3003  (spacing=200.0km, snap_to_F=40.0km)
[Gate-A v1] source counts:
source
C    2878
F     125
Name: count, dtype: int64
[Gate-A v1] per-ring gate count (eligible rings) describe:
count     187.000000
mean       16.058824
std        96.803158
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max      1124.000000
dtype: float64


,g_id,ring_id,lon,lat,x_m,y_m,source,s_target_km,ring_length_km,n_gate_plan
0,0,0,-40.914423,-21.863850,-8.176175e+06,1.657825e+07,C,0.000000,226034.691632,1130
1,1,0,-41.208990,-22.214714,-8.057071e+06,1.668939e+07,C,195.221029,226034.691632,1130
2,2,0,-41.416888,-22.303938,-7.884702e+06,1.679068e+07,C,390.442057,226034.691632,1130
3,3,0,-41.593772,-22.366944,-7.727442e+06,1.687828e+07,C,585.663086,226034.691632,1130
4,4,0,-41.796359,-22.458542,-7.552856e+06,1.697585e+07,C,780.884114,226034.691632,1130
5,5,0,-41.871856,-22.751072,-7.620812e+06,1.698162e+07,F,976.105143,226034.691632,1130
6,6,0,-42.101230,-23.094537,-7.541085e+06,1.706529e+07,C,1171.326172,226034.691632,1130
7,7,0,-42.238877,-23.054522,-7.358078e+06,1.714600e+07,C,1366.547200,226034.691632,1130
8,8,0,-42.369211,-23.035099,-7.193372e+06,1.721863e+07,C,1561.768229,226034.691632,1130
9,9,0,-42.520635,-23.032128,-7.010365e+06,1.729934e+07,C,1756.989257,226034.691632,1130


## 地圖檢查（把 F / Gate 分 layer）

In [31]:
import webbrowser
from pathlib import Path
import folium

min_lon, min_lat, max_lon, max_lat = BBOX_LL
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

m = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron")

# --- layers ---
fg_bbox   = folium.FeatureGroup(name="AOI bbox", show=True)
fg_rings  = folium.FeatureGroup(name="A2 coast rings (selected)", show=True)
fg_cnodes = folium.FeatureGroup(name="C nodes (sampled)", show=False)
fg_fnodes = folium.FeatureGroup(name="F nodes (feature points)", show=True)
fg_gates  = folium.FeatureGroup(name="Gate-A (coverage)", show=True)

# bbox
folium.Rectangle(bounds=[(min_lat, min_lon), (max_lat, max_lon)], fill=False).add_to(fg_bbox)
fg_bbox.add_to(m)

# helper: safe linestring drawing (normalize lon + split dateline)
def _norm_lon(lon): return ((lon + 180) % 360) - 180

def _split_dateline(latlons):
    segs, seg = [], []
    for lat, lon in latlons:
        if not seg:
            seg = [(lat, lon)]
            continue
        prev_lon = seg[-1][1]
        if abs(lon - prev_lon) > 180:
            if len(seg) >= 2:
                segs.append(seg)
            seg = [(lat, lon)]
        else:
            seg.append((lat, lon))
    if len(seg) >= 2:
        segs.append(seg)
    return segs

def add_linestring_ll_safe(line_m, group, step_km=20.0, weight=2, opacity=0.8):
    if line_m is None or line_m.is_empty:
        return
    pts = linestring_sample_points(line_m, step_m=max(1000.0, step_km * 1000.0))
    latlons = []
    for p in pts:
        lon, lat = proj.to_ll.transform(p.x, p.y)
        latlons.append((lat, _norm_lon(lon)))
    for seg in _split_dateline(latlons):
        folium.PolyLine(seg, weight=weight, opacity=opacity).add_to(group)

# pick rings to draw: prefer centroid inside AOI, choose top K by area
ring_by_id = {i: r for i, r in enumerate(coast_rings_smooth_m)}
def ring_centroid_ll(ring_m):
    c = ring_m.centroid
    lon, lat = proj.to_ll.transform(c.x, c.y)
    return lon, lat

if "centroid_lon" not in rings_df.columns:
    lons, lats = [], []
    for rid in rings_df["ring_id"]:
        lon, lat = ring_centroid_ll(ring_by_id[rid])
        lons.append(lon); lats.append(lat)
    rings_df["centroid_lon"] = lons
    rings_df["centroid_lat"] = lats

in_aoi = rings_df[
    (rings_df["centroid_lon"] >= min_lon) & (rings_df["centroid_lon"] <= max_lon) &
    (rings_df["centroid_lat"] >= min_lat) & (rings_df["centroid_lat"] <= max_lat)
].copy()

K = 60  # 想多看就加大
pick_df = (in_aoi if len(in_aoi) else rings_df).sort_values("area_km2", ascending=False).head(K)
rings_to_draw = [ring_by_id[rid] for rid in pick_df["ring_id"].tolist()]

for ring in rings_to_draw:
    add_linestring_ll_safe(ring, fg_rings, step_km=DRAW_STEP_KM, weight=2, opacity=0.85)
fg_rings.add_to(m)


# C nodes (sample) — BLUE
if len(C_nodes):
    sample_n = min(len(C_nodes), 3000)
    sample = C_nodes.sample(sample_n, random_state=0) if len(C_nodes) > sample_n else C_nodes
    for _, r in sample.iterrows():
        folium.CircleMarker(
            location=(r["lat"], r["lon"]),
            radius=2,
            color="blue",
            weight=1,
            fill=True,
            fill_color="blue",
            fill_opacity=0.45,
        ).add_to(fg_cnodes)
fg_cnodes.add_to(m)

# F nodes — ORANGE
if len(F_nodes):
    for _, r in F_nodes.iterrows():
        folium.CircleMarker(
            location=(r["lat"], r["lon"]),
            radius=4,
            color="orange",
            weight=2,
            fill=True,
            fill_color="orange",
            fill_opacity=0.85,
        ).add_to(fg_fnodes)
fg_fnodes.add_to(m)

# Gate-A — RED
if len(Gate_A):
    for _, r in Gate_A.iterrows():
        folium.CircleMarker(
            location=(r["lat"], r["lon"]),
            radius=6,
            color="red",
            weight=2,
            fill=True,
            fill_color="red",
            fill_opacity=0.9,
        ).add_to(fg_gates)

fg_gates.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

OUT_DIR = Path("./outputs_c_chain")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_html = OUT_DIR / "c_chain_F_gate_layers.html"
m.save(str(out_html))
print("Saved map to:", out_html.resolve())
webbrowser.open(out_html.resolve().as_uri())


Saved map to: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs_c_chain\c_chain_F_gate_layers.html


True